In [1]:
import pandas as pd
from textblob import TextBlob # TextBlob doesn't require a separate download for basic sentiment

In [2]:
df_songs = pd.read_csv('dataset.csv')

In [3]:
# 2. Extract Sentiment (Polarity: -1 to 1, Subjectivity: 0 to 1)
def get_sentiment(text):
    blob = TextBlob(str(text))
    return blob.sentiment.polarity, blob.sentiment.subjectivity

df_songs[['polarity', 'subjectivity']] = df_songs['lyrics_cleaned'].apply(
    lambda x: pd.Series(get_sentiment(x))
)

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split

# Use the columns created by NRCLex
X = df_songs[['polarity', 'subjectivity']]
y = df_songs['parent_genre']

model = LogisticRegression()
model.fit(X, y)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)
predictions = model.predict(X_test)

# 2. Get the overall accuracy score
accuracy = accuracy_score(y_test, predictions)
print(f"Overall Model Accuracy: {accuracy:.2%}")
print("-" * 30)

# 3. Print the full Summary (Precision, Recall, F1)
print("Classification Report:")
print(classification_report(y_test, predictions))

Overall Model Accuracy: 14.16%
------------------------------
Classification Report:
                  precision    recall  f1-score   support

       Asian Pop       0.05      0.01      0.02        89
       Classical       0.01      0.03      0.01        39
      Electronic       0.26      0.02      0.04       884
    Folk/Country       0.17      0.05      0.08       330
   Hip-Hop & R&B       0.00      0.00      0.00        24
    Jazz & Blues       0.00      0.00      0.00        49
           Latin       0.00      0.00      0.00        10
           Metal       0.23      0.60      0.33       479
             Pop       0.10      0.02      0.03       260
Reggae/Caribbean       0.04      0.22      0.07        87
            Rock       0.16      0.01      0.02       496
       Soul/Funk       0.11      0.01      0.01       185
  World/Regional       0.11      0.53      0.18       155

        accuracy                           0.14      3087
       macro avg       0.09      0.11      

In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 1. Define your feature groups
# Assume 'emotions_list' contains the column names you created with NRCLex
emotions_list = ['polarity', 'subjectivity']

# X now includes BOTH the text and the numeric emotion columns
X = df_songs[['lyrics_cleaned'] + emotions_list]
y = df_songs['parent_genre']

# 2. Split the data (using stratify to handle imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Create the ColumnTransformer
# This applies different processing to different columns
preprocessor = ColumnTransformer(
    transformers=[
        # Process text column with TF-IDF
        ('tfidf', TfidfVectorizer(max_features=5000, stop_words='english'), 'lyrics_cleaned'),
        # Process numeric emotion columns with a Scaler (very important for Logistic Regression)
        ('scaler', StandardScaler(), emotions_list)
    ]
)

# 4. Create the final pipeline
model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('clf', LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced'))
])

# 5. Train the model
print("Training model with combined features...")
model_pipeline.fit(X_train, y_train)

# 6. Make predictions and evaluate
predictions = model_pipeline.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, predictions):.2f}")
print("\nClassification Report:\n")
print(classification_report(y_test, predictions))

Training model with combined features...
Accuracy: 0.38

Classification Report:

                  precision    recall  f1-score   support

       Asian Pop       0.38      0.44      0.41        96
       Classical       0.09      0.32      0.14        44
      Electronic       0.54      0.26      0.35       882
    Folk/Country       0.44      0.55      0.49       322
   Hip-Hop & R&B       0.21      0.59      0.31        29
    Jazz & Blues       0.14      0.37      0.20        57
           Latin       0.07      0.33      0.12        12
           Metal       0.52      0.69      0.59       452
             Pop       0.21      0.25      0.23       285
Reggae/Caribbean       0.44      0.53      0.48        86
            Rock       0.30      0.15      0.20       493
       Soul/Funk       0.19      0.31      0.24       157
  World/Regional       0.64      0.67      0.66       172

        accuracy                           0.38      3087
       macro avg       0.32      0.42      0.34